# 06 — Прогін замороженої моделі на місті

Параметри: `city`, `model_version` (який файл моделі брати), `run_label` (`first_run` для першого запуску на C, `after_fix` після змін).

Робить те саме, що 05 для A/B, але **нічого не навчає**: бере модель з volume, ділить місто на train/test тим самим правилом,
рахує baseline на train цього міста й MAE на його test. Пише в ті самі gold-таблиці з окремим `run_label`,
тож дашборд і Genie бачать місто C поруч з A і B.

⚠️ На місті C цей ноутбук запускається **тільки як задача Job** після bronze → silver → 04 (це робить Людина 2 на другому тижні), з `model_version = v1` і без жодних змін у коді.
Для перевірки на Rome/Sicily ставте `dry_run = yes` — тоді в gold нічого не пишеться.

In [0]:
dbutils.widgets.text("city", "rome")
dbutils.widgets.text("model_version", "v1")
dbutils.widgets.text("run_label", "first_run")
dbutils.widgets.text("model_dir", "/Volumes/workspace/gold/models")
dbutils.widgets.dropdown("dry_run", "yes", ["no", "yes"])

city = dbutils.widgets.get("city").strip().lower()
if not city:
    raise ValueError("Вкажіть city явно (цей ноутбук запускає Job, а не ручний прогін)")
DRY_RUN = dbutils.widgets.get("dry_run") == "yes"
MODEL_VERSION = dbutils.widgets.get("model_version").strip()
RUN_LABEL = dbutils.widgets.get("run_label").strip()
MODEL_DIR = dbutils.widgets.get("model_dir").rstrip("/")

In [0]:
import os, sys, datetime
import numpy as np, pandas as pd, sklearn, joblib
from pyspark.sql import functions as F

sys.path.append(os.getcwd())
import model_utils as mu

CATALOG = "workspace"
FEATURES = f"{CATALOG}.gold.listing_features"
PREDICTIONS = f"{CATALOG}.gold.price_predictions"
METRICS = f"{CATALOG}.gold.model_metrics"

art = joblib.load(f"{MODEL_DIR}/price_model_{MODEL_VERSION}.joblib")
model = art["model"]
print(f"Модель {art['model_version']}: навчена {art['trained_at']} на {art['train_cities']}, sklearn {art['sklearn_version']}")
if art["sklearn_version"] != sklearn.__version__:
    print(f"⚠ sklearn зараз {sklearn.__version__}, а модель збережено в {art['sklearn_version']} — запишіть це у звіт")
if city in art["train_cities"]:
    print(f"⚠ {city} є серед міст навчання; для нього звітуйте числа з 05 (test), а не з цього прогону")

In [0]:
pdf = spark.table(FEATURES).filter(F.col("city") == city).toPandas()
if pdf.empty:
    raise ValueError(f"Немає ознак для '{city}' — запустіть bronze → silver → 04_gold_features")

pdf["in_model_scope"] = mu.in_scope(pdf)
scope = pdf[pdf["in_model_scope"]].copy()
scope["split"] = mu.host_split(scope)
pdf["split"] = scope["split"].reindex(pdf.index).fillna("out_of_scope")
train = scope[scope.split == "train"]
display(pdf.groupby("split").size().reset_index(name="rows"))

# baseline міста C — на ЙОГО train (те, що клієнт мав би без моделі); baseline_ab — правило з A+B
own_baseline = mu.fit_baseline(train)
pdf["baseline_pred"] = mu.predict_baseline(own_baseline, pdf)
pdf["baseline_ab_pred"] = mu.predict_baseline(art["baseline_ab"], pdf)
pdf["model_pred"] = mu.predict_model(model, pdf)

In [0]:
t = pdf[pdf["split"] == "test"]
metrics = pd.DataFrame([{"city": city, **mu.metrics_rows(t)}])
display(metrics)

## Що відрізняється від A/B (для пояснення розриву)

Частка пропусків і медіана кожної ознаки на C проти train A+B, плюс частка `property_type`, яких модель не бачила (вони йдуть у «infrequent»).

In [0]:
ref = spark.table(FEATURES).filter(F.col("city").isin(art["train_cities"]) & F.col("in_model_scope")).toPandas()
num = mu.NUM + mu.BOOL
cmp = pd.DataFrame({
    "missing_AB": ref[num].isna().mean(), "missing_C": scope[num].isna().mean(),
    "median_AB": ref[num].astype(float).median(), "median_C": scope[num].astype(float).median(),
    "mean_AB": ref[num].astype(float).mean(), "mean_C": scope[num].astype(float).mean(),
}).round(3)
display(cmp.reset_index().rename(columns={"index": "feature"}))

enc = model.named_steps["pre"].named_transformers_["cat"]
for col, cats in zip(mu.CAT, enc.categories_):
    ic = enc.infrequent_categories_[mu.CAT.index(col)]
    infreq = set(ic) if ic is not None else set()
    known = set(cats) - infreq
    share_unknown = (~scope[col].fillna("missing").isin(known)).mean()
    print(f"{col}: {share_unknown:.1%} оголошень у {city} мають значення, якого модель не знає окремо")
print("Медіана ціни: A+B =", ref.price.median(), f"| {city} =", scope.price.median())

In [0]:
def save(pdf_out, table, where):
    sdf = spark.createDataFrame(pdf_out)
    w = sdf.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        w.mode("overwrite").option("replaceWhere", where).saveAsTable(table)
    else:
        w.mode("overwrite").partitionBy("city").saveAsTable(table)

p = pdf[["city", "snapshot_date", "id", "host_id", "name", "listing_url", "neighbourhood_cleansed", "latitude", "longitude",
         "room_type", "property_type", "accommodates", "bedrooms", "in_model_scope", "split", "outlier_class",
         "price", "baseline_pred", "baseline_ab_pred", "model_pred"]].copy()
p["error"] = p["model_pred"] - p["price"]
p["abs_error"] = p["error"].abs()
p["pct_error"] = p["error"] / p["price"]
p["model_vs_actual"] = np.select(
    [p["price"].isna(), p["pct_error"] < -0.2, p["pct_error"] > 0.2],
    ["no actual price", "under-priced (model < actual by >20%)", "over-priced (model > actual by >20%)"],
    default="within 20%")
p["model_version"], p["run_label"] = MODEL_VERSION, RUN_LABEL
p["snapshot_date"] = pd.to_datetime(p["snapshot_date"]).dt.date
for c in ["price", "baseline_pred", "baseline_ab_pred", "model_pred", "error", "abs_error", "pct_error", "latitude",
          "longitude", "accommodates", "bedrooms"]:
    p[c] = p[c].astype("float64")
p["in_model_scope"] = p["in_model_scope"].astype(bool)
for c in ["name", "listing_url", "neighbourhood_cleansed", "room_type", "property_type", "split", "outlier_class"]:
    p[c] = p[c].astype(object).where(p[c].notna(), None)

where = f"model_version = '{MODEL_VERSION}' AND run_label = '{RUN_LABEL}' AND city = '{city}'"
if DRY_RUN:
    print("dry_run = yes: у gold нічого не записано")
else:
    save(p, PREDICTIONS, where)
    save(metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL, eval_set="test",
                        train_cities=",".join(art["train_cities"]), created_at=datetime.datetime.now()), METRICS, where)

## Звітна таблиця: три числа

Baseline на C, MAE моделі на A/B (з 05), MAE моделі на C — для кожного прогону, що вже є в `model_metrics`.

In [0]:
m = spark.table(METRICS).toPandas()
if DRY_RUN:
    m = pd.concat([m, metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL + " (dry run)",
                                     created_at=datetime.datetime.now())], ignore_index=True)
ab = m[(m.run_label == "train_test") & (m.city == "all")].set_index("model_version")["mae_model"]
c_runs = m[m.city == city].sort_values("created_at")
report = pd.DataFrame([{
    "run": f"{r.model_version} / {r.run_label}",
    f"baseline on {city} (own train)": r.mae_baseline,
    f"baseline A+B rule on {city}": r.mae_baseline_ab,
    "model MAE on A+B (test)": ab.get(r.model_version, np.nan),
    f"model MAE on {city} (test)": r.mae_model,
} for r in c_runs.itertuples()])
display(report)